# AtaRehber: evaluation on Kaggle

Runs `python -m evals.run_eval` on a Kaggle GPU, once per model of `MODELS` and per entry of `VARIANTS`, saves
the reports under `/kaggle/working/results`, and compares them in the last cell.

**Before running** (right-hand panel, *Session options*):
* **Accelerator:** GPU T4 x2 (or P100)
* **Internet:** On (needs a phone-verified Kaggle account)

Reports are kept only if you download them or run the notebook with **Save Version → Save & Run All**; files of an
interactive session are deleted when it ends.

In [ ]:
# Settings
REPO = "https://github.com/SirAlper/AtaRehber.git"
BRANCH = "main"
# Models to measure. qwen2.5:14b is about 9 GB and fits a 16 GB T4.
MODELS = ["qwen2.5:7b"]       # e.g. ["qwen2.5:7b", "qwen2.5:14b"]
# Settings to compare (names of .env.example). Every model runs once per entry; {} is the default behaviour.
VARIANTS = [{}]               # e.g. [{}, {"GRADER_MODE": "sentences"}]
GRADER_MODEL = ""             # model that checks the answers, e.g. "qwen2.5:7b"; empty: each model checks its own
DATASET = "university"        # "university" (127 questions, laws and regulations) or "demo" (51 questions)
STAGES = "all"                # "retrieval", "retrieval,routing", "all", "agents", "grader"
IDS = ""                      # e.g. "u-rek-01,u-rek-02" for a quick trial; empty: every question

In [ ]:
import sys
print(sys.version)
if sys.version_info < (3, 12):
    print("WARNING: the project is tested on Python 3.12+; this image is older and may fail.")
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 1. Code and Python packages

PyTorch is not installed: the image's own build (with CUDA) is used.

In [ ]:
%cd /kaggle/working
!rm -rf AtaRehber && git clone --depth 1 --branch {BRANCH} {REPO} AtaRehber
%cd /kaggle/working/AtaRehber
!git log --oneline -1
!pip install -q -r requirements.txt

## 2. Ollama and the language models

In [ ]:
import subprocess, time, urllib.request

!apt-get install -y -qq zstd > /dev/null 2>&1
!curl -fsSL https://ollama.com/install.sh | sh > /dev/null 2>&1
server = subprocess.Popen(["ollama", "serve"], stdout=open("/kaggle/working/ollama.log", "w"), stderr=subprocess.STDOUT)
for _ in range(60):
    try:
        urllib.request.urlopen("http://localhost:11434/api/tags", timeout=2)
        print("Ollama is running")
        break
    except Exception:
        time.sleep(1)
else:
    raise RuntimeError("Ollama did not start; see /kaggle/working/ollama.log")
for model in dict.fromkeys(MODELS + ([GRADER_MODEL] if GRADER_MODEL else [])):
    print("Pulling", model)
    !ollama pull {model} 2>&1 | tail -1

## 3. Embedding and reranker models (about 3.3 GB)

In [ ]:
!python download_model.py 2>&1 | tail -5

## 4. Evaluation

Each question prints a line when it starts and one when it is done, with the progress and the time left. The
runs follow one another; `ollama ps` after each run shows whether the model was fully on the GPU.

In [ ]:
import os

base_env = dict(os.environ)
args = f"--stages {STAGES} --out /kaggle/working/results"
if DATASET == "university":
    args += " --dataset evals/dataset_university.jsonl --corpus evals/corpus_university --no-database"
if IDS:
    args += f" --ids {IDS}"
runs = [(model, variant) for model in MODELS for variant in VARIANTS]
# Models added to the settings after section 2 ran are pulled here (a model already there takes seconds)
for model in dict.fromkeys(MODELS + ([GRADER_MODEL] if GRADER_MODEL else [])):
    !ollama pull {model} 2>&1 | tail -1
for number, (model, variant) in enumerate(runs, 1):
    os.environ.clear()
    os.environ.update(base_env)
    os.environ.update({"OLLAMA_MODEL": model, "OLLAMA_GRADER_MODEL": GRADER_MODEL, "RAG_DEVICE": "cuda"})
    if DATASET == "university":
        # The threshold measured for article-aware chunks (docs/evaluation.md)
        os.environ["RAG_MIN_RERANKER_SCORE"] = "0.05"
    os.environ.update({key: str(value) for key, value in variant.items()})
    print(f"\n######## Run {number}/{len(runs)}: {model} {variant or '(default settings)'} ########")
    print("python -m evals.run_eval", args)
    !python -m evals.run_eval {args} 2>&1 | grep -v -E "^\[20|Batches|Loading weights"
    !ollama ps
    if model != GRADER_MODEL and (number == len(runs) or runs[number][0] != model):
        # Free the GPU memory for the next model
        !ollama stop {model}

## 5. The reports side by side

In [ ]:
import glob, json

paths = sorted(glob.glob("/kaggle/working/results/report_*.json"))[-len(runs):]
if not paths:
    raise SystemExit("No report was written; see the output above.")
reports = [json.load(open(path, encoding="utf-8")) for path in paths]
labels = [f"run {number}" for number in range(1, len(reports) + 1)]
for label, path, report, (model, variant) in zip(labels, paths, reports, runs[-len(reports):]):
    settings = report["config"]
    print(f"{label}: {settings['llm_model']} {variant or '(default settings)'}")
    print(f"   checked by {settings.get('grader_model', settings['llm_model'])} | commit {report['git_commit']}"
          f" | {report['case_count']} cases | {path}")

width = 12
for stage in reports[0]["summary"]:
    print(f"\n== {stage} ==")
    print(" " * 36 + "".join(f"{label:>{width}}" for label in labels))
    for key, value in reports[0]["summary"][stage].items():
        if isinstance(value, (dict, list)):
            continue
        values = [report["summary"].get(stage, {}).get(key) for report in reports]
        shown = [f"{v:.3f}" if isinstance(v, float) else str(v) for v in values]
        print(f"  {key:<34}" + "".join(f"{text:>{width}}" for text in shown))